---
title: "Orbital refinement for excited states of H4 using state-averaged DMRG"
author:
    - name: Thuy Truong
date: "2026-03-10"
categories: [code]
image: "preview.png"
image-width: "1cm"
image-height: "1cm"
format:
    html:
        toc: true
        code-fold: false
        eval: true
code-annotations: hover
execute:
    freeze: false
    eval: true
    cache: false
---

In this tutorial, we will explore the process of orbital refinement for excited states of the H4 molecule by using the NWChem orbitals as guess orbitals and performing state-averaged DMRG calculations. First, the parameters for the multiwavelet representation are defined.

In [ ]:
molecule_name = "h4"
n_elec = 4  #<1>
number_roots = 3  #<2>
iterations = 3  #<3>
box_size = 50.0  #<4>
wavelet_order = 7  #<5>
madness_thresh = 0.0001  #<6>
basisset = "6-31g"  #<7>

1. Number of electrons
2. Number of states (ground state, 1st excited state, 2nd excited state)
3. Number of iterations for orbital refinement
4. Size of the simulation box
5. Order of wavelet basis functions
6. Threshold for numerical precision of function representation
7. Initial basis set for calculation

### Run NWChem calculation

To get the initial guess orbitals, we will perform a NWChem calculation. If you are using the FrayedEnds devcontainer or the singularity image, NWChem is already installed. Otherwise, you will need to install NWChem and **adjust the path** in the code below.

In [ ]:
import subprocess as sp

nwchem_input = (
    """
title "molecule"
memory stack 1500 mb heap 100 mb global 1400 mb
charge 0
geometry units angstrom noautosym nocenter
    H 0.0 0.0 -1.5
    H 0.0 0.0 -0.5
    H 0.0 0.0 0.5
    H 0.0 0.0 1.5
end
basis
  * library """
    + basisset
    + """
end
scf
 maxiter 200
end
task scf
"""
)

with open("nwchem", "w") as f:  #<1>
    f.write(nwchem_input)   #<1>
programm = sp.call(
    "/opt/anaconda3/envs/frayedends/bin/nwchem nwchem", #<2>
    stdout=open("nwchem.out", "w"),
    stderr=open("nwchem_err.log", "w"),
    shell=True,
)   #<1>

1. Run NWChem calculation
2. Adjust the path to NWChem here if you are not using the FrayedEnds devcontainer or the singularity image.

### Convert NWChem AOs and MOs to MRA-Orbitals
Next, we will read the molecular orbitals (MOs) from the NWChem calculation and translate them into multiwavelets by using the NWChem_Converter class in FrayedEnds.

In [ ]:
import frayedends as fe

world = fe.MadWorld3D(L=box_size, k=wavelet_order, thresh=madness_thresh) #<1>

converter = fe.NWChem_Converter(world) #<2>
converter.read_nwchem_file("nwchem") #<3>
orbs = converter.get_mos() #<4>
Vnuc = converter.get_Vnuc() #<5>
nuclear_repulsion_energy = converter.get_nuclear_repulsion_energy() #<6>

n_orbitals = len(orbs)
for i in range(n_orbitals): #<7>
    orbs[i].type = "active" #<7>

molecule = fe.MolecularGeometry(units="angstrom")
molecule.add_atom(0.0, 0.0, -1.5, "H")
molecule.add_atom(0.0, 0.0, -0.5, "H")
molecule.add_atom(0.0, 0.0, 0.5, "H")
molecule.add_atom(0.0, 0.0, 1.5, "H")

for i in range(n_orbitals):
    world.cube_plot(f"initial_orb{i}", orbs[i], molecule)

1. Create a frayedends world object with the specified parameters
2. Create a NWChem_Converter object
3. Read the NWChem output file to extract the orbitals and other relevant information
4. Get the molecular orbitals (MOs) from the converter
5. Get the nuclear potential from the converter
6. Get the nuclear repulsion energy from the converter
7. Set the type of all orbitals to "active" for the DMRG calculation

### Calculate initial integrals

After obtaining the orbitals, we can calculate the initial integrals required for the DMRG calculation, including the two-body, kinetic, potential, and overlap integrals.

In [ ]:
integrals = fe.Integrals3D(world) #<1>
G = integrals.compute_two_body_integrals(orbs, ordering="chem").elems #<2>
T = integrals.compute_kinetic_integrals(orbs)  #<2>
V = integrals.compute_potential_integrals(orbs, Vnuc) #<2>
S = integrals.compute_overlap_integrals(orbs)  #<2>

1. Create an integrals object to compute the initial integrals
2. Compute the two-body, kinetic, potential, and overlap integrals using the orbitals obtained from NWChem

### Perform state-averaged DMRG calculation and extract rdms

Now we will perform a state-averaged DMRG calculation using the integrals obtained from the previous step and extract the one-body and two-body reduced density matrices (rdms).

In [ ]:
import numpy as np
from pyblock2.driver.core import DMRGDriver, SymmetryTypes

driver = DMRGDriver(scratch="./tmp", symm_type=SymmetryTypes.SU2, n_threads=8) #<1>
driver.initialize_system(n_sites=n_orbitals, n_elec=n_elec, spin=0) #<1>
mpo = driver.get_qc_mpo(h1e=T + V, g2e=G, ecore=nuclear_repulsion_energy, iprint=0) #<1>
ket = driver.get_random_mps(tag="KET", bond_dim=100, nroots=number_roots) #<1>
energies = driver.dmrg(mpo, ket, n_sweeps=10, bond_dims=[100], noises=[1e-5] * 4 + [0], thrds=[1e-10] * 8, iprint=1) #<1>
print("State-averaged MPS energies = [%s]" % " ".join("%20.15f" % x for x in energies)) #<1>

kets = [driver.split_mps(ket, ir, tag="KET-%d" % ir) for ir in range(ket.nroots)]
sa_1pdm = np.mean([driver.get_1pdm(k) for k in kets], axis=0)  #<2>
sa_2pdm = np.mean([driver.get_2pdm(k) for k in kets], axis=0).transpose( #<3>
    0, 3, 1, 2 #<3>
)  #<3>
print(
    "Energy from SA-pdms = %20.15f"
    % (np.einsum("ij,ij->", sa_1pdm, T + V) + 0.5 * np.einsum("ijkl,ijkl->", sa_2pdm, G) + nuclear_repulsion_energy)
)
sa_2pdm_phys = sa_2pdm.swapaxes(1, 2)


1. Performe State Average (SA) DMRG calculation and extract rdms
2. Compute the state-average 1-body reduced density matrix
3. Compute the state average 2-body reduced density matrix

### Orbital refinement

Finally, we will perform the orbital refinement by using the state-averaged 1-body and 2-body reduced density matrices obtained from the DMRG calculation. The refined orbitals can then be used for further state-averaged DMRG calculations to improve the accuracy of the excited state energies. The orbital refinement is repeated for a specified number of iterations defined at the beginning of the tutorial.

In [ ]:
import time

for iter in range(iterations):
    iter_start = time.perf_counter()

    opti = fe.Optimization3D(world, Vnuc, nuclear_repulsion_energy) #<1>
    orbs = opti.get_orbitals(orbitals=orbs, rdm1=sa_1pdm, rdm2=sa_2pdm_phys, opt_thresh=0.001, occ_thresh=0.001) #<1>

    for i in range(n_orbitals):
        world.cube_plot(f"orb{i}", orbs[i], molecule)  #<2>

    G = integrals.compute_two_body_integrals(orbs, ordering="chem").elems  #<3>
    T = integrals.compute_kinetic_integrals(orbs)  #<3>
    V = integrals.compute_potential_integrals(orbs, Vnuc) #<3>
    S = integrals.compute_overlap_integrals(orbs) #<3>

    driver = DMRGDriver(scratch="./tmp", symm_type=SymmetryTypes.SU2, n_threads=8) #<4>
    driver.initialize_system(n_sites=n_orbitals, n_elec=n_elec, spin=0) #<4>
    mpo = driver.get_qc_mpo(h1e=T + V, g2e=G, ecore=nuclear_repulsion_energy, iprint=0) #<4>
    ket = driver.get_random_mps(tag="KET", bond_dim=100, nroots=number_roots) #<4>
    energies = driver.dmrg(mpo, ket, n_sweeps=10, bond_dims=[100], noises=[1e-5] * 4 + [0], thrds=[1e-10] * 8, iprint=1) #<4>
    print("State-averaged MPS energies after refinement = [%s]" % " ".join("%20.15f" % x for x in energies))

    kets = [driver.split_mps(ket, ir, tag="KET-%d" % ir) for ir in range(ket.nroots)]
    sa_1pdm = np.mean([driver.get_1pdm(k) for k in kets], axis=0)  #<5>
    sa_2pdm = np.mean([driver.get_2pdm(k) for k in kets], axis=0).transpose( #<6>
        0, 3, 1, 2 #<6>
    )  #<6>
    print(
        "Energy from SA-pdms = %20.15f"
        % (np.einsum("ij,ij->", sa_1pdm, T + V) + 0.5 * np.einsum("ijkl,ijkl->", sa_2pdm, G) + nuclear_repulsion_energy)
    )
    sa_2pdm_phys = sa_2pdm.swapaxes(1, 2)

fe.cleanup(globals())

1. Create an optimization object for orbital refinement and get the refined orbitals using the state-averaged 1-body and 2-body rdms
2. Plot the refined orbitals
3. Calculate the integrals with the refined orbitals
4. Perform a state-averaged DMRG calculation with the refined orbitals and print the energies after refinement
5. Compute the state-average 1-body rdm with the refined orbitals
6. Compute the state-average 2-body rdm with the refined orbitals

The orbital refinement process is repeated for the specified number of iterations.
At the end of the orbital refinement, the energies of the ground state and excited states should be improved compared to the initial DMRG calculation with the NWChem orbitals as guess orbitals.

The initial orbitals obtained from NWChem and the refined orbitals after the orbital optimization can be visualized using the cube files and plotted using py3Dmol.

In [ ]:
#| echo: false

import py3Dmol

initial_orb = open("initial_orb1.cube").read()
refined_orb = open("orb1.cube").read()

v = py3Dmol.view(width=800, height=400, viewergrid=(1,2))

v.addVolumetricData(initial_orb, "cube",
                    {"isoval": -0.001, "color": "red", "opacity": 0.75},
                    viewer=(0,0))
v.addVolumetricData(initial_orb, "cube",
                    {"isoval": 0.001, "color": "blue", "opacity": 0.75},
                    viewer=(0,0))
v.addModel(initial_orb, "cube", viewer=(0,0))
v.setStyle({"sphere":{}}, viewer=(0,0))

v.addVolumetricData(refined_orb, "cube",
                    {"isoval": -0.001, "color": "red", "opacity": 0.75},
                    viewer=(0,1))
v.addVolumetricData(refined_orb, "cube",
                    {"isoval": 0.001, "color": "blue", "opacity": 0.75},
                    viewer=(0,1))
v.addModel(refined_orb, "cube", viewer=(0,1))
v.setStyle({"sphere":{}}, viewer=(0,1))

v.zoomTo()
v.show()